# SDIC revision analyses (Google Colab)

Per-image analyses requested in the PAA review of the SDIC manuscript. This is the Colab version of `revision_analyses.py`: the same methods and outputs, plus Google Drive input and output, a demo on simulated data, and a light mode for repeated fold partitions.

**How to use**

1. Export the per-image NLL tables from the original notebooks (section 0).
2. Set the options in cell 1, then choose *Runtime > Run all*.
3. Read the report (cell 11). Per-comparison tables are in cell 12; cell 13 downloads everything as a zip.
4. Copy each result into the matching PENDING marker of the response letter (table at the end).

**With Google Drive**: save the exported files in one Drive folder, keep `DATA_SOURCE = files` and point `INPUT_GLOB` at them. Results are written to `OUT_DIR` on Drive.

**Without Google Drive**: download the exported files to your computer, untick `USE_GOOGLE_DRIVE`, set `DATA_SOURCE = upload`, and choose the files when cell 9 asks. Results go to `/content/revision_results` and cell 13 downloads them as a zip. Colab deletes `/content` when the runtime stops, so download the zip before closing the notebook.

To try the notebook first, set `DATA_SOURCE` to `demo`: every analysis then runs on simulated data.

**Model** (manuscript Eq. 1): $d_{if} = \mu + a_f + b_i + e_{if}$, where $d_{if}$ is the paired per-image NLL difference (arm A minus arm B) for image $i$ under test family $f$, averaged over severity levels, exactly as `paired_matrix()` builds it.

| Analysis | Review point |
|---|---|
| Crossed components, exact F test, exact CI for the SE ratio, generalized CI for crossed $\rho_{family}$, full crossed SE | R2.2, R2.5, R3.1 |
| Table 5 quantities recomputed from per-image data | R2.1 |
| Permutation test of Section 6.5, Hotelling's $T^2$, studentized permutation test | R2.3, R3.2 |
| Leave one family out, with SE ratios | R2.6 |
| Within-fold correlation; spread of the mean over repeated fold partitions | R2.6, R3.5 |
| Components at each severity level; families matched on the probe's accuracy loss | R1.3, R3.4 |
| Oracle-class arm C5o, once added to Phase 0.11 | R2.6 |

**Input**: one row per (image, test family, severity, arm); CSV, CSV.GZ or Parquet, with columns `dataset, backbone, run, fold, image_id, family, severity, arm, nll` and, for amplitude matching, `correct` (1 when the probe's prediction is right). Optional rows with `family == "clean"` and `severity == 0` anchor the amplitude interpolation at zero damage. `run` separates calibration grids and fold partitions: `G1` for Phase 0.11, `G2` for Phases 1 to 4, `G1-split3` for grid G1 with the fourth fold partition.

## 0. Export the per-image tables (run this in the original notebooks)

The original notebooks compute per-image NLLs but do not save them. Add the lines below and run. Then either copy the files into one Drive folder, for example `MyDrive/SDIC/per_image/`, or download them to your computer by adding, after each `to_csv`:

```python
from google.colab import files
files.download(str(OUT / "per_image_nll_G1.csv.gz"))   # use the file name just written
```

Each file is a few megabytes. Adapt variable names where a notebook differs.

**Grid G1** (source of Tables 3 to 5). The simplest route is `PHASE0_11_family_uncertainty_revision.ipynb`: it runs the original notebook, saves these tables, adds the oracle arm and the fold partitions, and runs the analyses of this notebook itself. To export by hand instead, add this cell at the end of `PHASE0_11_family_uncertainty.ipynb`, after section 5:

```python
import pandas as pd
n = [len(f) for f in FOLDS]
fold = np.concatenate([np.full(len(SEVERITIES) * nk, k) for k, nk in enumerate(n)])
sev = np.concatenate([np.repeat(np.asarray(SEVERITIES), nk) for nk in n])
rows = [pd.DataFrame(dict(dataset="NEU-CLS", backbone=BACKBONE, run="G1", fold=fold, image_id=d["im"],
                          family=fam, severity=sev, arm=arm.split()[0], nll=d["nll"],
                          correct=(d["p"].argmax(1) == d["y"]).astype("int8")))
        for (arm, fam), d in POOLF.items()]
pd.concat(rows).to_csv(OUT / "per_image_nll_G1.csv.gz", index=False)
```

The section 3 loop concatenates the five folds in `POOLF`, each fold severity-major, which is what `fold` and `sev` reproduce. `correct` uses the calibrated argmax, which equals the probe's argmax because every arm rescales the logits by a positive temperature; the analysis checks that it is the same for every arm.

*Oracle-class arm (R2.6).* In `gather()` add

```python
HO = np.zeros((len(L), NC), np.float32); HO[np.arange(len(L)), Y] = 1.0
```

return it as `ho=HO`, and add to `ARM_SPECS`:

```python
"C5o linear / true-class one-hot": ("scalar", LinearT, "ho"),
```

This arm sees the true class at test time, so it is an upper bound on what class information alone can do, not a deployable calibrator. Comparisons against it (`C4 vs C5o` and so on) are added automatically.

**Grid G2**, `paired_matrix()` in `PHASE4_crossed_effects`, `PHASE3_1`, `PHASE1` and `PHASE2_1`. Define `ROWS = []` before the fold loop and replace the loop over `TEST_FAMILIES` with:

```python
for fi, fam in enumerate(TEST_FAMILIES):
    Lt, Qt, Ht, Yt = gather(te, [(fam, s) for s in SEVERITIES]); ft = {"q": Qt, "h": Ht}
    for nm, (mdl, f_) in arms.items():
        with torch.no_grad():
            p = softmax(mdl(torch.as_tensor(Lt, dtype=torch.float32),
                            torch.as_tensor(ft[f_], dtype=torch.float32)).numpy())
        ROWS.append(pd.DataFrame(dict(
            dataset=DATASET, backbone=BACKBONE, run="G2", fold=k,
            image_id=np.tile(te, len(SEVERITIES)), family=fam,
            severity=np.repeat(SEVERITIES, len(te)), arm=nm, nll=nll_pi(p, Yt),
            correct=(Lt.argmax(1) == Yt).astype("int8"))))
```

After the loop: `pd.concat(ROWS).to_csv(OUT / f"per_image_nll_G2_{BACKBONE}.csv.gz", index=False)`. Rows are severity-major within a fold, as in `gather()`.

**Repeated fold partitions (R3.5).** Rerun an export with only the fold split changed: `StratifiedKFold(5, shuffle=True, random_state=SEED + 1000 * r)`, where `r = 0` reproduces the original split. Name the runs `G1-split0` to `G1-split9`, or keep the original as `G1` and name the others `G1-split1` to `G1-split9`; both work. With `LIGHT_SPLITS` on, only the original partition gets the full analyses, and the others contribute their means and standard errors to the repeated-partition table.

In [ ]:
#@title 1. Settings { display-mode: "form" }
#@markdown **Data.** `files` reads every file matching `INPUT_GLOB`; `upload` opens a file picker; `demo` simulates data.
DATA_SOURCE = "files"  #@param ["files", "upload", "demo"]
#@markdown Untick `USE_GOOGLE_DRIVE` to work without Drive: use `upload`, and results go to `/content/revision_results`.
USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
INPUT_GLOB = "/content/drive/MyDrive/SDIC/per_image/per_image_nll_*.csv.gz"  #@param {type:"string"}
OUT_DIR = "/content/drive/MyDrive/SDIC/revision_results"  #@param {type:"string"}
#@markdown **Comparisons.** Empty runs the twelve of Table 5, plus each quality arm against C5o when present. Otherwise list them, e.g. `C4:C2, C4:C5o`.
PAIRS = ""  #@param {type:"string"}
#@markdown **Precision.** `QUICK` divides the resampling counts by ten for a first pass.
QUICK = False  #@param {type:"boolean"}
ALPHA = 0.05  #@param {type:"number"}
N_PERM = 20000  #@param {type:"integer"}
N_PERM_WTS = 5000  #@param {type:"integer"}
N_GPQ = 20000  #@param {type:"integer"}
N_BOOT = 2000  #@param {type:"integer"}
SEED = 20261101  #@param {type:"integer"}
#@markdown **Amplitude matching.** Targets are accuracy losses, e.g. `0.05, 0.10`; empty chooses them automatically.
AMP_METHOD = "interp"  #@param ["interp", "nearest"]
AMP_TARGETS = ""  #@param {type:"string"}
#@markdown **Repeated fold partitions.** Runs whose name contains `SPLIT_TAG` are partitions of the same grid.
SPLIT_TAG = "-split"  #@param {type:"string"}
LIGHT_SPLITS = True  #@param {type:"boolean"}

In [ ]:
#@title 2. Set up { display-mode: "form" }
import glob, json, math, os, shutil, sys, tempfile, time
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import scipy
from scipy import stats
from IPython.display import display, Markdown

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kw):
        return x

try:
    from google.colab import drive as colab_drive, files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def _on_drive(p):
    return str(p).startswith("/content/drive")

if _on_drive(OUT_DIR) and not (USE_GOOGLE_DRIVE and IN_COLAB):
    OUT_DIR = "/content/revision_results" if IN_COLAB else str(Path.cwd() / "revision_results")
    print(f"Results go to {OUT_DIR}" + ("; download them with cell 13 before the runtime stops." if IN_COLAB else "."))
if DATA_SOURCE == "files" and _on_drive(INPUT_GLOB) and not USE_GOOGLE_DRIVE:
    raise ValueError("USE_GOOGLE_DRIVE is off but INPUT_GLOB points to Drive: set DATA_SOURCE = upload, "
                     "or put the files in /content and point INPUT_GLOB there")
if IN_COLAB and USE_GOOGLE_DRIVE and (_on_drive(OUT_DIR) or (DATA_SOURCE == "files" and _on_drive(INPUT_GLOB))):
    colab_drive.mount("/content/drive")

_pairs = [tuple(s.strip() for s in p.split(":")) for p in PAIRS.split(",") if p.strip()]
if any(len(p) != 2 or not all(p) for p in _pairs):
    raise ValueError(f"PAIRS must look like 'C4:C2, C4:C5o', got {PAIRS!r}")
_scale = 10 if QUICK else 1
CFG = SimpleNamespace(
    alpha=float(ALPHA), seed=int(SEED), pairs=_pairs,
    n_perm=max(int(N_PERM) // _scale, 500), n_perm_wts=max(int(N_PERM_WTS) // _scale, 300),
    n_gpq=max(int(N_GPQ) // _scale, 2000), n_boot=max(int(N_BOOT) // _scale, 300),
    targets=[float(t) for t in AMP_TARGETS.replace(";", ",").split(",") if t.strip()] or None,
    amp_method=AMP_METHOD, split_tag=SPLIT_TAG, light_splits=bool(LIGHT_SPLITS),
    out_dir=OUT_DIR, demo=(DATA_SOURCE == "demo"))
print(f"Python {sys.version.split()[0]}, numpy {np.__version__}, scipy {scipy.__version__}, "
      f"pandas {pd.__version__}; running in Colab: {IN_COLAB}")
for k, v in vars(CFG).items():
    print(f"  {k:13s} {v}")

In [ ]:
#@title 3. Library: data { display-mode: "form" }
REQUIRED = ["dataset", "backbone", "run", "fold", "image_id", "family", "severity", "arm", "nll"]
TREATMENTS = ["C4", "C6c", "C7", "C7q", "C8", "C8b"]   # the quality-conditioned arms of Table 5
BASELINES = ["C2", "C5"]
DEFAULT_ALPHA, DEFAULT_SEED, NAN = 0.05, 20261101, float("nan")


def validate_long(df):
    """Check columns and duplicates; shorten long arm names ("C4 linear / raw q" -> "C4")."""
    missing = [c for c in REQUIRED if c not in df.columns]
    if missing:
        raise ValueError(f"input is missing columns {missing}; see section 0")
    df = df.copy()
    df["arm"] = df["arm"].astype(str).str.split().str[0]
    for c in ("dataset", "backbone", "run", "family"):
        df[c] = df[c].astype(str)
    dup = df.duplicated(["dataset", "backbone", "run", "image_id", "family", "severity", "arm"])
    if dup.any():
        raise ValueError(f"{int(dup.sum())} duplicated (run, image, family, severity, arm) rows")
    return df


def load_long(paths):
    """Read one or more long per-image tables (CSV, CSV.GZ or Parquet)."""
    frames = [pd.read_parquet(p) if str(p).endswith(".parquet") else pd.read_csv(p) for p in paths]
    return validate_long(pd.concat(frames, ignore_index=True))


def inventory(df):
    """One row per dataset / backbone / run, listing arms whose image x family x severity design is incomplete."""
    rows = []
    for key, g in df.groupby(["dataset", "backbone", "run"], sort=False):
        test = g[g["family"] != "clean"]
        n_img, n_fam, n_sev = test["image_id"].nunique(), test["family"].nunique(), test["severity"].nunique()
        per_arm = test.groupby("arm").size()
        full = {a for a, n in per_arm.items() if n == n_img * n_fam * n_sev}
        rows.append(dict(dataset=key[0], backbone=key[1], run=key[2], images=n_img, families=n_fam,
                         severities=n_sev, folds=test["fold"].nunique(), arms=" ".join(sorted(per_arm.index)),
                         incomplete_arms=" ".join(sorted(set(per_arm.index) - full)) or "none",
                         has_correct="correct" in g.columns, has_clean_rows=bool((g["family"] == "clean").any())))
    return pd.DataFrame(rows)


def paired_cube(g, A, B):
    """Complete image x family x severity cube of nll[A] - nll[B] for one dataset / backbone / run."""
    arms = set(g["arm"].unique())
    if A not in arms or B not in arms:
        return None
    test = g[g["family"] != "clean"]
    fams = list(pd.unique(test["family"]))
    sevs = np.sort(test["severity"].unique())
    imgs = np.sort(test["image_id"].unique())
    a = test[test["arm"] == A].set_index(["image_id", "family", "severity"])["nll"]
    b = test[test["arm"] == B].set_index(["image_id", "family", "severity"])["nll"]
    full = pd.MultiIndex.from_product([imgs, fams, sevs], names=["image_id", "family", "severity"])
    d = (a - b).reindex(full)
    if d.isna().any():
        raise ValueError(f"{A} vs {B}: {int(d.isna().sum())} of {len(d)} image x family x severity "
                         "cells are missing; the crossed analysis needs the complete design")
    D3 = d.to_numpy().reshape(len(imgs), len(fams), len(sevs))

    if (test.groupby("image_id")["fold"].nunique() > 1).any():
        raise ValueError("an image appears in more than one test fold")
    folds = test.groupby("image_id")["fold"].first().reindex(imgs).to_numpy()

    out = dict(D3=D3, families=fams, severities=sevs, images=imgs, folds=folds,
               D_clean=None, acc=None, acc_clean=None, warnings=[])
    clean = g[g["family"] == "clean"]
    if len(clean):
        ca = clean[clean["arm"] == A].set_index("image_id")["nll"].reindex(imgs)
        cb = clean[clean["arm"] == B].set_index("image_id")["nll"].reindex(imgs)
        if not (ca.isna().any() or cb.isna().any()):
            out["D_clean"] = (ca - cb).to_numpy()
    if "correct" in g.columns:
        sub = g[g["arm"].isin([A, B])]
        varies = sub.groupby(["image_id", "family", "severity"])["correct"].nunique()
        if (varies > 1).any():
            out["warnings"].append(f"'correct' differs between {A} and {B} in {int((varies > 1).sum())} "
                                   f"cells, so some arm changes predictions; accuracy is taken from {B}")
        cb_ = test[test["arm"] == B].set_index(["image_id", "family", "severity"])["correct"].reindex(full)
        if not cb_.isna().any():
            out["acc"] = cb_.to_numpy().reshape(len(imgs), len(fams), len(sevs)).mean(0)
        if len(clean):
            cc = clean[clean["arm"] == B].set_index("image_id")["correct"].reindex(imgs)
            if not cc.isna().any():
                out["acc_clean"] = float(cc.mean())
    return out

In [ ]:
#@title 4. Library: crossed model { display-mode: "form" }
def mean_squares(D):
    """Two-way ANOVA mean squares of a complete m x F matrix (images x families)."""
    D = np.asarray(D, float)
    m, F = D.shape
    gm = D.mean(); rm = D.mean(1); cm = D.mean(0)
    MSA = m * ((cm - gm) ** 2).sum() / (F - 1)
    MSB = F * ((rm - gm) ** 2).sum() / (m - 1)
    res = D - rm[:, None] - cm[None, :] + gm
    MSE = (res ** 2).sum() / ((m - 1) * (F - 1))
    return m, F, gm, MSA, MSB, MSE


def _satterthwaite(terms):
    """terms: list of (coefficient, mean square, df). Returns (value, df)."""
    v = sum(c * ms for c, ms, _ in terms)
    den = sum((c * ms) ** 2 / nu for c, ms, nu in terms)
    return v, (v ** 2 / den if den > 0 else float("inf"))


def two_way(D, alpha=DEFAULT_ALPHA, n_gpq=20000, seed=DEFAULT_SEED):
    """Crossed two-way random-effects analysis of a complete m x F matrix.

    E[MS_A] = s_e^2 + m s_a^2, E[MS_B] = s_e^2 + F s_b^2, E[MS_E] = s_e^2.
    SE_family^2 = MS_A/(mF), SE_image^2 = MS_B/(mF), SE ratio R = sqrt(MS_A/MS_B), and
    R^2 (s_e^2 + F s_b^2)/(s_e^2 + m s_a^2) ~ F(F-1, m-1): an exact CI for the true ratio.
    The generalized CI for rho_family follows Burdick, Borror and Montgomery (2005)."""
    m, F, gm, MSA, MSB, MSE = mean_squares(D)
    nA, nB, nE = F - 1, m - 1, (m - 1) * (F - 1)
    s2a = max((MSA - MSE) / m, 0.0); s2b = max((MSB - MSE) / F, 0.0); s2e = MSE
    tot = s2a + s2b + s2e
    se_fam = math.sqrt(MSA / (m * F)); se_img = math.sqrt(MSB / (m * F))
    R = se_fam / se_img if se_img > 0 else float("inf")
    fq = lambda p, a, b: stats.f.ppf(p, a, b)
    r_lo = R / math.sqrt(fq(1 - alpha / 2, nA, nB)); r_hi = R / math.sqrt(fq(alpha / 2, nA, nB))
    FA = MSA / MSE
    lam_lo = max((FA / fq(1 - alpha / 2, nA, nE) - 1) / m, 0.0)
    lam_hi = max((FA / fq(alpha / 2, nA, nE) - 1) / m, 0.0)

    # two-component quantities (Table 5 definitions): s_e^2 = mF SE_image^2 = MS_B and
    # s_a^2 = var(family means) - s_e^2/m = (MS_A - MS_B)/m; the SE ratio maps one to one to rho
    s2a_2c = max((MSA - MSB) / m, 0.0); s2e_2c = MSB
    rho_2c = s2a_2c / (s2a_2c + s2e_2c) if s2a_2c + s2e_2c > 0 else 0.0
    g = lambda th: max(((th - 1) / m) / (1 + (th - 1) / m), 0.0)

    # variance of the grand mean over all three sources (truncated moment estimates)
    terms = []
    if MSA > MSE: terms.append((1.0, MSA, nA))
    if MSB > MSE: terms.append((1.0, MSB, nB))
    k = 1 - int(MSA > MSE) - int(MSB > MSE)
    if k: terms.append((float(k), MSE, nE))
    v_full, df_full = _satterthwaite(terms)
    se_full = math.sqrt(max(v_full, 0.0) / (m * F))
    t_full = stats.t.ppf(1 - alpha / 2, df_full) if np.isfinite(df_full) else stats.norm.ppf(1 - alpha / 2)
    t_fam = stats.t.ppf(1 - alpha / 2, nA); z = stats.norm.ppf(1 - alpha / 2)

    out = dict(m=m, F=F, mean=gm, MS_A=MSA, MS_B=MSB, MS_E=MSE,
               sigma_a=math.sqrt(s2a), sigma_b=math.sqrt(s2b), sigma_e=math.sqrt(s2e),
               rho_family=s2a / tot if tot > 0 else 0.0, rho_image=s2b / tot if tot > 0 else 0.0,
               F_family=FA, p_F_family=stats.f.sf(FA, nA, nE),
               var_ratio_a_e_lo=lam_lo, var_ratio_a_e_hi=lam_hi,
               se_family=se_fam, se_image=se_img, se_ratio=R, se_ratio_lo=r_lo, se_ratio_hi=r_hi,
               p_se_ratio=stats.f.sf(R ** 2, nA, nB),
               sigma_a_2c=math.sqrt(s2a_2c), sigma_e_2c=math.sqrt(s2e_2c), rho_2c=rho_2c,
               rho_2c_lo=g(r_lo ** 2), rho_2c_hi=g(r_hi ** 2),
               se_full=se_full, df_full=df_full, full_over_family=se_full / se_fam if se_fam > 0 else NAN,
               ci_family=(gm - t_fam * se_fam, gm + t_fam * se_fam),
               ci_image=(gm - z * se_img, gm + z * se_img),
               ci_full=(gm - t_full * se_full, gm + t_full * se_full))
    if n_gpq:
        rng = np.random.default_rng(seed)
        GE = nE * MSE / rng.chisquare(nE, n_gpq)
        GA = nA * MSA / rng.chisquare(nA, n_gpq)
        GB = nB * MSB / rng.chisquare(nB, n_gpq)
        ga = np.maximum((GA - GE) / m, 0.0); gb = np.maximum((GB - GE) / F, 0.0)
        rho = ga / (ga + gb + GE)
        out["rho_family_lo"], out["rho_family_hi"] = np.quantile(rho, [alpha / 2, 1 - alpha / 2])
    return out


def table5_row(tw):
    """The four quantities printed in Table 5, from a two_way() result."""
    return dict(sigma_a=tw["sigma_a_2c"], sigma_e=tw["sigma_e_2c"], rho=tw["rho_2c"],
                se_ratio=tw["se_ratio"], se_ratio_lo=tw["se_ratio_lo"], se_ratio_hi=tw["se_ratio_hi"])


def image_bootstrap(D, B=2000, alpha=DEFAULT_ALPHA, seed=DEFAULT_SEED):
    """Percentile bootstrap over images, as in the manuscript (families held fixed)."""
    r = np.asarray(D, float).mean(1); m = len(r)
    rng = np.random.default_rng(seed)
    means = np.empty(B)
    for s in range(0, B, 500):
        b = min(500, B - s)
        means[s:s + b] = r[rng.integers(0, m, (b, m))].mean(1)
    lo, hi = np.quantile(means, [alpha / 2, 1 - alpha / 2])
    return dict(se_boot=float(means.std(ddof=1)), ci_boot=(float(lo), float(hi)),
                se_from_width=float((hi - lo) / (2 * stats.norm.ppf(1 - alpha / 2))))

In [ ]:
#@title 5. Library: tests of the family effect { display-mode: "form" }
def _permuted(D, rng, b):
    m, F = D.shape
    idx = np.argsort(rng.random((b, m, F)), axis=2)
    return np.take_along_axis(np.broadcast_to(D, (b, m, F)), idx, axis=2)


def perm_test(D, n_perm=20000, seed=DEFAULT_SEED, batch=500):
    """Section 6.5 test: variance of the family means, family labels permuted within images.

    Exact under H0: within each image, the joint law of (d_i1, ..., d_iF) is invariant under
    permutation of the family index, images independent. Family-specific variances violate
    this, which is why the two tests below are also reported."""
    D = np.asarray(D, float)
    rng = np.random.default_rng(seed)
    obs = float(D.mean(0).var(ddof=1))
    null = np.empty(n_perm)
    for s in range(0, n_perm, batch):
        b = min(batch, n_perm - s)
        null[s:s + b] = _permuted(D, rng, b).mean(1).var(axis=1, ddof=1)
    med = float(np.median(null))
    return dict(stat=obs, null_median=med, ratio_to_median=obs / med if med > 0 else float("inf"),
                p=float((1 + np.sum(null >= obs)) / (n_perm + 1)), n_perm=n_perm)


def _contrast(F):
    return np.hstack([np.eye(F - 1), -np.ones((F - 1, 1))])


def _t2(P):
    """Hotelling T^2 for H0: equal family means, for a stack P of shape (b, m, F)."""
    b, m, F = P.shape
    Y = P @ _contrast(F).T
    yb = Y.mean(1)
    Yc = Y - yb[:, None, :]
    S = np.einsum("bmi,bmj->bij", Yc, Yc) / (m - 1)
    try:
        sol = np.linalg.solve(S, yb[..., None])[..., 0]
    except np.linalg.LinAlgError:
        sol = np.einsum("bij,bj->bi", np.linalg.pinv(S), yb)
    return m * np.einsum("bi,bi->b", yb, sol)


def hotelling_rm(D):
    """Repeated-measures Hotelling T^2: equal family means under any within-image covariance
    (no sphericity, no equal variances). Exact under multivariate normality of the image
    vectors; the chi-square reference is its large-m (Wald-type statistic) limit."""
    D = np.asarray(D, float)
    m, F = D.shape
    T2 = float(_t2(D[None])[0])
    Fst = (m - F + 1) / ((m - 1) * (F - 1)) * T2
    return dict(T2=T2, F=Fst, df1=F - 1, df2=m - F + 1, p_F=float(stats.f.sf(Fst, F - 1, m - F + 1)),
                p_chi2=float(stats.chi2.sf(T2, F - 1)))


def wts_perm_test(D, n_perm=5000, seed=DEFAULT_SEED, batch=250):
    """Studentized permutation test (Friedrich, Brunner and Pauly 2017): the Wald-type statistic
    recomputed, covariance included, on every within-image permutation. Exact when the
    within-image exchangeability of perm_test holds, and asymptotically valid for equal family
    means under any covariance when it does not (Chung and Romano 2013)."""
    D = np.asarray(D, float)
    rng = np.random.default_rng(seed + 1)
    obs = float(_t2(D[None])[0])
    null = np.empty(n_perm)
    for s in range(0, n_perm, batch):
        b = min(batch, n_perm - s)
        null[s:s + b] = _t2(_permuted(D, rng, b))
    return dict(T2=obs, p=float((1 + np.sum(null >= obs)) / (n_perm + 1)), n_perm=n_perm)

In [ ]:
#@title 6. Library: sensitivity analyses { display-mode: "form" }
def lofo(D, families, alpha=DEFAULT_ALPHA):
    """Leave one family out: family-level t test, image-level z test, SE ratio and F test."""
    D = np.asarray(D, float)
    rows = []
    for j in [None] + list(range(D.shape[1])):
        Dj = D if j is None else np.delete(D, j, axis=1)
        tw = two_way(Dj, alpha=alpha, n_gpq=0)
        t = stats.ttest_1samp(Dj.mean(0), 0.0)
        zi = tw["mean"] / tw["se_image"]
        rows.append(dict(omitted="none" if j is None else families[j], n_families=Dj.shape[1],
                         mean=tw["mean"], t_family=float(t.statistic), p_family=float(t.pvalue),
                         ci_family_lo=tw["ci_family"][0], ci_family_hi=tw["ci_family"][1],
                         p_image=float(2 * stats.norm.sf(abs(zi))),
                         se_ratio=tw["se_ratio"], se_ratio_lo=tw["se_ratio_lo"], se_ratio_hi=tw["se_ratio_hi"],
                         F_family=tw["F_family"], p_F_family=tw["p_F_family"]))
    return pd.DataFrame(rows)


_FOLD_KEYS = ("images_per_fold", "icc_fold", "icc_fold_lo", "icc_fold_hi", "F_fold", "p_fold", "var_fold",
              "var_fold_hi", "design_effect", "design_effect_hi", "se_image_iid", "se_image_deff",
              "se_image_deff_hi", "se_image_cluster_robust", "se_image_two_stage")


def fold_clustered(D, folds, alpha=DEFAULT_ALPHA, B=2000, seed=DEFAULT_SEED):
    """Does sharing the probe and calibrators within a fold inflate the image-level SE?

    One-way ANOVA of the image means on fold gives the within-fold intraclass correlation and
    the design effect 1 + (n0 - 1) ICC; a cluster-robust SE and a two-stage bootstrap (folds,
    then images within folds) are cross-checks. With five folds all of them are rough, which
    is why the repeated-partition table is the more informative check."""
    D = np.asarray(D, float)
    r = D.mean(1); m = len(r)
    ks = np.unique(folds); K = len(ks)
    if K < 2:
        return dict(n_folds=K, **{k: NAN for k in _FOLD_KEYS})
    groups = [r[folds == k] for k in ks]
    nk = np.array([len(x) for x in groups])
    gm = r.mean(); mk = np.array([x.mean() for x in groups])
    MSb = (nk * (mk - gm) ** 2).sum() / (K - 1)
    MSw = sum(((x - x.mean()) ** 2).sum() for x in groups) / (m - K)
    n0 = (m - (nk ** 2).sum() / m) / (K - 1)
    s2f = max((MSb - MSw) / n0, 0.0); icc = s2f / (s2f + MSw)
    Fo = MSb / MSw
    FL = Fo / stats.f.ppf(1 - alpha / 2, K - 1, m - K); FU = Fo / stats.f.ppf(alpha / 2, K - 1, m - K)
    icc_lo = max((FL - 1) / (FL + n0 - 1), 0.0); icc_hi = max((FU - 1) / (FU + n0 - 1), 0.0)
    se_iid = r.std(ddof=1) / math.sqrt(m)
    u = np.array([(x - gm).sum() for x in groups])
    se_cr = math.sqrt(K / (K - 1) * (u ** 2).sum()) / m
    rng = np.random.default_rng(seed + 2)
    bm = np.empty(B)
    for b in range(B):
        pick = rng.integers(0, K, K)
        bm[b] = np.concatenate([groups[c][rng.integers(0, nk[c], nk[c])] for c in pick]).mean()
    deff = 1 + (n0 - 1) * icc; deff_hi = 1 + (n0 - 1) * icc_hi
    s2f_hi = icc_hi * MSw / (1 - icc_hi) if icc_hi < 1 else float("inf")
    return dict(n_folds=K, images_per_fold=n0, icc_fold=icc, icc_fold_lo=icc_lo, icc_fold_hi=icc_hi,
                F_fold=Fo, p_fold=float(stats.f.sf(Fo, K - 1, m - K)),
                var_fold=s2f / K, var_fold_hi=s2f_hi / K,       # added to the variance of the grand mean
                design_effect=deff, design_effect_hi=deff_hi,
                se_image_iid=se_iid, se_image_deff=se_iid * math.sqrt(deff),
                se_image_deff_hi=se_iid * math.sqrt(deff_hi),
                se_image_cluster_robust=se_cr, se_image_two_stage=float(bm.std(ddof=1)))


def split_stem(run, tag):
    """'G1-split3' -> 'G1'; a run without the tag is its own stem."""
    return run.split(tag)[0] if tag and tag in run else run


def split_variability(rows, tag="-split"):
    """How far does each mean move when only the fold partition changes?

    Partitions of one grid are the runs '<grid><tag><k>' plus the base run '<grid>' (counted as
    partition 0 unless '<grid><tag>0' also exists, which would be the same split twice). The SD
    over partitions measures training-sample variability that the image-level interval ignores;
    it is a lower bound, because the partitions share most training images (Bates, Hastie and
    Tibshirani 2024)."""
    df = pd.DataFrame(rows)
    if df.empty or not tag:
        return None
    df["grid"] = [split_stem(r, tag) for r in df["run"]]
    tagged = df["run"].str.contains(tag, regex=False)
    stems = set(zip(df.loc[tagged, "dataset"], df.loc[tagged, "backbone"], df.loc[tagged, "grid"]))
    if not stems:
        return None
    runs = set(zip(df["dataset"], df["backbone"], df["run"]))
    keep = []
    for i, r in df.iterrows():
        if (r["dataset"], r["backbone"], r["grid"]) not in stems:
            continue
        if tag in r["run"] or (r["dataset"], r["backbone"], f"{r['grid']}{tag}0") not in runs:
            keep.append(i)
    out = []
    for (ds, bb, gr, comp), g in df.loc[keep].groupby(["dataset", "backbone", "grid", "comparison"], sort=False):
        if len(g) < 2:
            continue
        sd = float(g["mean"].std(ddof=1))
        se_i, se_f = float(g["se_image"].mean()), float(g["se_family"].mean())
        out.append(dict(dataset=ds, backbone=bb, grid=gr, comparison=comp, n_splits=len(g),
                        mean_of_means=float(g["mean"].mean()), sd_across_splits=sd, se_image=se_i,
                        se_family=se_f, sd_over_se_image=sd / se_i, sd_over_se_family=sd / se_f,
                        se_ratio_min=float(g["se_ratio"].min()), se_ratio_max=float(g["se_ratio"].max())))
    return pd.DataFrame(out) if out else None


def per_severity(D3, severities, alpha=DEFAULT_ALPHA, n_gpq=20000, seed=DEFAULT_SEED):
    """Crossed components separately at each severity level, plus the severity average."""
    D3 = np.asarray(D3, float)
    rows = []
    labels = [str(s) for s in severities] + ["mean"]
    for j, s in enumerate(labels):
        tw = two_way(D3[:, :, j] if s != "mean" else D3.mean(2), alpha=alpha, n_gpq=n_gpq, seed=seed)
        rows.append(dict(severity=s, mean=tw["mean"], sigma_a=tw["sigma_a"], sigma_b=tw["sigma_b"],
                         sigma_e=tw["sigma_e"], rho_family=tw["rho_family"],
                         rho_family_lo=tw.get("rho_family_lo"), rho_family_hi=tw.get("rho_family_hi"),
                         se_ratio=tw["se_ratio"], se_ratio_lo=tw["se_ratio_lo"], se_ratio_hi=tw["se_ratio_hi"],
                         F_family=tw["F_family"], p_F_family=tw["p_F_family"]))
    return pd.DataFrame(rows)


def _locate(loss, tau):
    """First crossing of tau on a severity ladder of losses -> (j0, j1, w) or None."""
    for j in range(1, len(loss)):
        lo, hi = loss[j - 1], loss[j]
        if lo <= tau <= hi and hi > lo:
            return j - 1, j, (tau - lo) / (hi - lo)
    return None


def amplitude_matched(D3, acc, families, severities, targets=None, D_clean=None, acc_clean=None,
                      use_clean=False, method="interp", alpha=DEFAULT_ALPHA, n_gpq=20000, seed=DEFAULT_SEED,
                      min_families=3):
    """Re-estimate the family component with families compared at matched damage.

    Damage is the probe's accuracy loss, which no arm changes (all arms rescale logits by a
    positive temperature), so the matching variable is not the analysed NLL difference.
    method="interp": for each family, interpolate the per-image differences linearly between
    the two severity levels whose losses bracket the target; families whose ladder does not
    reach the target are left out at that target (at least min_families are needed).
    method="nearest": use each family's level closest to the target and report the mismatch.
    use_clean=True also interpolates toward the clean condition (zero loss). It is off by
    default: the clean column is the same for every family, so mixing it in shrinks family
    differences mechanically at low targets. Default targets: accuracy losses 0.05 to 0.40."""
    D3 = np.asarray(D3, float); acc = np.asarray(acc, float)
    m, F, S = D3.shape
    base = acc_clean if acc_clean is not None else float(np.max(acc))
    loss = base - acc                                   # F x S, larger = more damage
    levels = list(severities)
    if use_clean and D_clean is not None and acc_clean is not None:
        loss = np.hstack([np.zeros((F, 1)), loss])
        D3 = np.concatenate([np.asarray(D_clean, float)[:, None, None].repeat(F, 1), D3], axis=2)
        levels = [0] + levels
    if targets is None:
        targets = [0.05, 0.10, 0.20, 0.30, 0.40]
    rows = []
    for tau in targets:
        cols, used, where, mism = [], [], [], []
        for f in range(F):
            if method == "nearest":
                j = int(np.argmin(np.abs(loss[f] - tau)))
                cols.append(D3[:, f, j]); used.append(families[f]); where.append(str(levels[j]))
                mism.append(float(loss[f, j] - tau))
                continue
            hit = _locate(loss[f], tau)
            if hit is None:
                continue
            j0, j1, w = hit
            cols.append((1 - w) * D3[:, f, j0] + w * D3[:, f, j1]); used.append(families[f])
            where.append(f"{levels[j0]}+{w:.2f}")
        row = dict(target_loss=float(tau), n_families=len(cols), families=";".join(used),
                   position=";".join(where))
        if method == "nearest":
            row["max_abs_mismatch"] = float(np.max(np.abs(mism)))
        if len(cols) >= min_families:
            tw = two_way(np.column_stack(cols), alpha=alpha, n_gpq=n_gpq, seed=seed)
            row.update(mean=tw["mean"], sigma_a=tw["sigma_a"], sigma_b=tw["sigma_b"], sigma_e=tw["sigma_e"],
                       rho_family=tw["rho_family"], rho_family_lo=tw.get("rho_family_lo"),
                       rho_family_hi=tw.get("rho_family_hi"), se_ratio=tw["se_ratio"],
                       se_ratio_lo=tw["se_ratio_lo"], se_ratio_hi=tw["se_ratio_hi"],
                       F_family=tw["F_family"], p_F_family=tw["p_F_family"])
        rows.append(row)
    return pd.DataFrame(rows), loss

In [ ]:
#@title 7. Library: driver, report and simulated data { display-mode: "form" }
def default_pairs(arms):
    """The twelve comparisons of Table 5 that the data contain, plus each quality arm against C5o."""
    pairs = [(a, b) for b in BASELINES for a in TREATMENTS if a in arms and b in arms]
    oracle = sorted(x for x in arms if x.startswith("C5") and x.endswith("o"))
    pairs += [(a, o) for o in oracle for a in TREATMENTS if a in arms]
    return pairs


def primary_runs(keys, tag):
    """Per dataset / backbone / grid, the run that gets the full analyses: the base run ('G1')
    if present, else '<grid><tag>0', else the first repeated partition."""
    by = {}
    for ds, bb, run in keys:
        by.setdefault((ds, bb, split_stem(run, tag)), []).append(run)
    prim = {}
    for (ds, bb, stem), runs in by.items():
        zero = f"{stem}{tag}0"
        prim[(ds, bb, stem)] = stem if stem in runs else (zero if zero in runs else sorted(runs)[0])
    return prim


def _div(a, b):
    return a / b if b and np.isfinite(b) else NAN


def analyse_pair(cube, A, B, cfg, light=False):
    """Every analysis for one comparison. Light mode (repeated partitions) keeps only the mean,
    the standard errors and the fold check."""
    D3 = cube["D3"]; D = D3.mean(2); fams = cube["families"]
    tw = two_way(D, alpha=cfg.alpha, n_gpq=0 if light else cfg.n_gpq, seed=cfg.seed)
    fc = fold_clustered(D, cube["folds"], alpha=cfg.alpha, B=200 if light else cfg.n_boot, seed=cfg.seed)
    bt = dict(se_from_width=NAN, ci_boot=(NAN, NAN)); pm = dict(p=NAN, ratio_to_median=NAN)
    t2 = dict(T2=NAN, p_F=NAN); wt = dict(p=NAN); lf = ps = am = None
    if not light:
        bt = image_bootstrap(D, B=cfg.n_boot, alpha=cfg.alpha, seed=cfg.seed)
        pm = perm_test(D, n_perm=cfg.n_perm, seed=cfg.seed)
        t2 = hotelling_rm(D)
        wt = wts_perm_test(D, n_perm=cfg.n_perm_wts, seed=cfg.seed)
        lf = lofo(D, fams, alpha=cfg.alpha)
        ps = per_severity(D3, cube["severities"], alpha=cfg.alpha, n_gpq=cfg.n_gpq, seed=cfg.seed)
        if cube["acc"] is not None:
            am, _ = amplitude_matched(D3, cube["acc"], fams, cube["severities"], targets=cfg.targets,
                                      D_clean=cube["D_clean"], acc_clean=cube["acc_clean"],
                                      method=cfg.amp_method, alpha=cfg.alpha, n_gpq=cfg.n_gpq, seed=cfg.seed)
    lofo_rows = lf.iloc[1:] if lf is not None else None
    row = dict(comparison=f"{A} vs {B}", light=light, m=tw["m"], F=tw["F"], S=D3.shape[2], mean=tw["mean"],
               se_family=tw["se_family"], se_image=tw["se_image"], se_image_boot=bt["se_from_width"],
               se_ratio=tw["se_ratio"], se_ratio_lo=tw["se_ratio_lo"], se_ratio_hi=tw["se_ratio_hi"],
               p_se_ratio=tw["p_se_ratio"],
               t5_sigma_a=tw["sigma_a_2c"], t5_sigma_e=tw["sigma_e_2c"], t5_rho=tw["rho_2c"],
               t5_rho_lo=tw["rho_2c_lo"], t5_rho_hi=tw["rho_2c_hi"],
               sigma_a=tw["sigma_a"], sigma_b=tw["sigma_b"], sigma_e=tw["sigma_e"],
               rho_family=tw["rho_family"], rho_family_lo=tw.get("rho_family_lo", NAN),
               rho_family_hi=tw.get("rho_family_hi", NAN), F_family=tw["F_family"], p_F_family=tw["p_F_family"],
               se_full=tw["se_full"], df_full=tw["df_full"], full_over_family=tw["full_over_family"],
               ci_family_lo=tw["ci_family"][0], ci_family_hi=tw["ci_family"][1],
               ci_full_lo=tw["ci_full"][0], ci_full_hi=tw["ci_full"][1],
               ci_image_boot_lo=bt["ci_boot"][0], ci_image_boot_hi=bt["ci_boot"][1],
               perm_p=pm["p"], perm_ratio_to_median=pm["ratio_to_median"],
               T2=t2["T2"], T2_p_F=t2["p_F"], wts_perm_p=wt["p"],
               icc_fold=fc["icc_fold"], icc_fold_hi=fc["icc_fold_hi"], design_effect=fc["design_effect"],
               design_effect_hi=fc["design_effect_hi"],
               se_ratio_fold_adjusted=_div(tw["se_family"], fc["se_image_deff"]),
               se_ratio_fold_adjusted_conservative=_div(tw["se_family"], fc["se_image_deff_hi"]),
               se_image_cluster_robust=fc["se_image_cluster_robust"],
               se_image_two_stage=fc["se_image_two_stage"],
               fold_share_of_family_var=_div(fc["var_fold"], tw["se_family"] ** 2),
               fold_share_of_family_var_hi=_div(fc["var_fold_hi"], tw["se_family"] ** 2),
               se_full_with_fold=math.sqrt(tw["se_full"] ** 2 + fc["var_fold"]) if np.isfinite(fc["var_fold"]) else NAN,
               se_full_with_fold_hi=math.sqrt(tw["se_full"] ** 2 + fc["var_fold_hi"]) if np.isfinite(fc["var_fold_hi"]) else NAN,
               lofo_min_p_family=float(lofo_rows["p_family"].min()) if lofo_rows is not None else NAN,
               lofo_family_at_min=str(lofo_rows.loc[lofo_rows["p_family"].idxmin(), "omitted"]) if lofo_rows is not None else "",
               lofo_min_se_ratio=float(lofo_rows["se_ratio"].min()) if lofo_rows is not None else NAN)
    tabs = dict(lofo=lf, per_severity=ps, amplitude_matched=am,
                family_by_severity_means=pd.DataFrame(D3.mean(0), index=fams,
                                                      columns=[str(s) for s in cube["severities"]]))
    return row, tabs


def _f(x, d=3):
    try:
        return f"{x:.{d}f}" if x is not None and np.isfinite(x) else "n/a"
    except TypeError:
        return "n/a"


def _p(x):
    if x is None or not np.isfinite(x):
        return "n/a"
    return f"{x:.1e}" if x < 1e-3 else f"{x:.3f}"


def write_report(groups, cfg, split=None):
    """Markdown report: one section per dataset / backbone / run, then the repeated partitions."""
    L = ["# Revision analyses", ""]
    if getattr(cfg, "demo", False):
        L += ["**Demo on simulated data. These numbers are not results.**", ""]
    L += [f"alpha {cfg.alpha}; {cfg.n_perm} permutations (plain), {cfg.n_perm_wts} (studentized); "
          f"{cfg.n_gpq} generalized-pivot draws; {cfg.n_boot} bootstrap replicates; seed {cfg.seed}.", ""]
    for key, rows, warns in groups:
        if not rows:
            if warns:
                L += [f"## {' / '.join(key)}", ""] + [f"Warning: {w}" for w in warns] + [""]
            continue
        L += [f"## {' / '.join(key)}", ""]
        if all(r["light"] for r in rows):
            L += ["Repeated fold partition, light mode: only the mean, the standard errors and the fold check "
                  "were computed. See the repeated-partition table at the end.", ""]
            L += [f"Warning: {w}" for w in warns] + ([""] if warns else [])
            continue
        L += [f"m = {rows[0]['m']} images, F = {rows[0]['F']} families, {rows[0]['S']} severity levels.", ""]
        L += [f"Warning: {w}" for w in warns] + ([""] if warns else [])
        L += ["### Table 5 quantities, recomputed (two-component, Section 5.3 definitions)", "",
              "| comparison | sigma_a | sigma_e | rho [95% exact] | SE ratio [95% exact] | SE ratio, bootstrap SE |",
              "|---|---|---|---|---|---|"]
        for r in rows:
            L.append(f"| {r['comparison']} | {_f(r['t5_sigma_a'])} | {_f(r['t5_sigma_e'])} | {_f(r['t5_rho'])} "
                     f"[{_f(r['t5_rho_lo'])}, {_f(r['t5_rho_hi'])}] | {_f(r['se_ratio'], 2)} "
                     f"[{_f(r['se_ratio_lo'], 2)}, {_f(r['se_ratio_hi'], 2)}] | "
                     f"{_f(_div(r['se_family'], r['se_image_boot']), 2)} |")
        L += ["", "### Crossed decomposition and tests of the family effect", "",
              "| comparison | sigma_a | sigma_b | sigma_e | rho_family [95% GCI] | F test p | permutation p "
              "(obs/median) | Hotelling p | studentized perm p | full SE / family SE |",
              "|---|---|---|---|---|---|---|---|---|---|"]
        for r in rows:
            L.append(f"| {r['comparison']} | {_f(r['sigma_a'])} | {_f(r['sigma_b'])} | {_f(r['sigma_e'])} | "
                     f"{_f(r['rho_family'])} [{_f(r['rho_family_lo'])}, {_f(r['rho_family_hi'])}] | "
                     f"{_p(r['p_F_family'])} | {_p(r['perm_p'])} ({_f(r['perm_ratio_to_median'], 1)}) | "
                     f"{_p(r['T2_p_F'])} | {_p(r['wts_perm_p'])} | {_f(r['full_over_family'], 4)} |")
        L += ["", "### Intervals for the mean difference", "",
              "| comparison | mean | image bootstrap 95% | family t 95% | full crossed 95% (df) |",
              "|---|---|---|---|---|"]
        for r in rows:
            L.append(f"| {r['comparison']} | {_f(r['mean'], 4)} | [{_f(r['ci_image_boot_lo'], 4)}, "
                     f"{_f(r['ci_image_boot_hi'], 4)}] | [{_f(r['ci_family_lo'], 4)}, {_f(r['ci_family_hi'], 4)}] "
                     f"| [{_f(r['ci_full_lo'], 4)}, {_f(r['ci_full_hi'], 4)}] ({_f(r['df_full'], 1)}) |")
        L += ["", "### Leave one family out and fold dependence", "",
              "| comparison | smallest LOFO family p (family omitted) | smallest LOFO SE ratio | fold ICC [upper] "
              "| design effect [upper] | SE ratio, fold-adjusted [conservative] |",
              "|---|---|---|---|---|---|"]
        for r in rows:
            L.append(f"| {r['comparison']} | {_p(r['lofo_min_p_family'])} ({r['lofo_family_at_min']}) | "
                     f"{_f(r['lofo_min_se_ratio'], 2)} | {_f(r['icc_fold'], 4)} [{_f(r['icc_fold_hi'], 4)}] | "
                     f"{_f(r['design_effect'], 2)} [{_f(r['design_effect_hi'], 2)}] | "
                     f"{_f(r['se_ratio_fold_adjusted'], 2)} [{_f(r['se_ratio_fold_adjusted_conservative'], 2)}] |")
        L += ["", "Fold component as a share of the family-level variance: "
              + "; ".join(f"{r['comparison']} {_f(r['fold_share_of_family_var'], 3)} "
                          f"[upper {_f(r['fold_share_of_family_var_hi'], 3)}]" for r in rows) + ".", "",
              "Per-comparison tables (leave one family out, per severity, amplitude matched, family x severity "
              "means) are in the comparison folders of the output directory.", ""]
    if split is not None and len(split):
        L += ["## Repeated fold partitions", "",
              "| dataset / backbone / grid | comparison | partitions | SD of the mean over partitions | SE image | "
              "SE family | SD / SE image | SD / SE family | SE ratio range |", "|---|---|---|---|---|---|---|---|---|"]
        for _, r in split.iterrows():
            L.append(f"| {r['dataset']} / {r['backbone']} / {r['grid']} | {r['comparison']} | {r['n_splits']} | "
                     f"{_f(r['sd_across_splits'], 4)} | {_f(r['se_image'], 4)} | {_f(r['se_family'], 4)} | "
                     f"{_f(r['sd_over_se_image'], 2)} | {_f(r['sd_over_se_family'], 2)} | "
                     f"{_f(r['se_ratio_min'], 2)} to {_f(r['se_ratio_max'], 2)} |")
        L.append("")
    return "\n".join(L)


def run_all(df, cfg):
    """Run every comparison in every dataset / backbone / run; write CSVs and report.md to cfg.out_dir."""
    out = Path(cfg.out_dir); out.mkdir(parents=True, exist_ok=True)
    tag = cfg.split_tag

    def order(k):                     # base run first, then its partitions in numeric order
        stem = split_stem(k[2], tag)
        tail = k[2][len(stem) + len(tag):] if tag and tag in k[2] else ""
        return (k[0], k[1], stem, k[2] != stem, int(tail) if tail.isdigit() else -1, k[2])

    keys = sorted(df[["dataset", "backbone", "run"]].drop_duplicates().itertuples(index=False, name=None), key=order)
    prim = primary_runs(keys, tag)
    grouped = df.groupby(["dataset", "backbone", "run"], sort=False)
    jobs = []
    for key in keys:
        g = grouped.get_group(key)
        pairs = cfg.pairs or default_pairs(set(g["arm"].unique()))
        light = (cfg.light_splits and bool(cfg.split_tag) and cfg.split_tag in key[2]
                 and prim[(key[0], key[1], split_stem(key[2], cfg.split_tag))] != key[2])
        jobs += [(key, g, A, B, light) for A, B in pairs]
    groups = {k: [] for k in keys}; warns = {k: set() for k in keys}; rows = []; tables = {}
    for key, g, A, B, light in tqdm(jobs, desc="comparisons"):
        try:
            cube = paired_cube(g, A, B)
        except ValueError as e:
            warns[key].add(str(e)); continue
        if cube is None:
            warns[key].add(f"{A} vs {B}: an arm is missing in this run"); continue
        warns[key].update(cube["warnings"])
        row, tabs = analyse_pair(cube, A, B, cfg, light=light)
        row.update(dataset=key[0], backbone=key[1], run=key[2])
        groups[key].append(row); rows.append(row); tables[(key, row["comparison"])] = tabs
        d = out / "_".join(key).replace("/", "-") / f"{A}_vs_{B}"
        d.mkdir(parents=True, exist_ok=True)
        for name, tab in tabs.items():
            if tab is not None:
                tab.to_csv(d / f"{name}.csv", index=(name == "family_by_severity_means"))
    summary = pd.DataFrame(rows)
    if len(summary):
        cols = ["dataset", "backbone", "run"] + [c for c in summary.columns if c not in ("dataset", "backbone", "run")]
        summary = summary[cols]
        summary.to_csv(out / "summary.csv", index=False)
    split = split_variability(rows, cfg.split_tag) if rows else None
    if split is not None:
        split.to_csv(out / "split_variability.csv", index=False)
    report = write_report([(k, groups[k], sorted(warns[k])) for k in keys], cfg, split)
    (out / "report.md").write_text(report)
    (out / "settings.json").write_text(json.dumps(vars(cfg), indent=2, default=str))
    return dict(summary=summary, tables=tables, split=split, report=report, out=out)


DEMO_FAMILIES = ["motion_blur", "shot_noise", "brightness_drift", "contrast_loss", "vignetting", "vibration_jitter"]


def simulate_long_table(runs=("G1",), arms=("C2", "C4", "C5"), m=900, n_sev=5, seed=0, clean=True,
                        dataset="SIM", backbone="toy"):
    """A simulated per-image NLL table in the export format, for the demo and the self-test.
    Families have unequal severity ladders; each arm adds its own family and image effects."""
    rng = np.random.default_rng(seed)
    F = len(DEMO_FAMILIES); sevs = np.arange(1, n_sev + 1)
    loss = np.linspace(0.02, 0.09, F)[:, None] * sevs[None, :]          # F x S accuracy loss
    b = np.abs(rng.normal(0, 0.2, m))                                    # image difficulty
    fam_eff = {a: (0.0 if a == arms[0] else 1.0) * rng.normal(0, 0.05, F) for a in arms}
    img_eff = {a: rng.normal(0, 0.03, m) for a in arms}
    shift = {a: (0.0 if a == arms[0] else rng.uniform(-0.04, 0.02)) for a in arms}
    correct = (rng.random((m, F, n_sev)) < (0.95 - loss)[None]).astype(int)
    correct_clean = (rng.random(m) < 0.95).astype(int)
    frames = []
    for run in runs:
        folds = rng.permutation(np.arange(m) % 5)
        base = 0.3 + 1.2 * loss[None] + b[:, None, None] + rng.normal(0, 0.15, (m, F, n_sev))
        for a in arms:
            nll = (base + shift[a] + fam_eff[a][None, :, None] + img_eff[a][:, None, None]
                   + rng.normal(0, 0.05, (m, F, n_sev)))
            frames.append(pd.DataFrame(dict(
                dataset=dataset, backbone=backbone, run=run, fold=np.repeat(folds, F * n_sev),
                image_id=np.repeat(np.arange(m), F * n_sev), family=np.tile(np.repeat(DEMO_FAMILIES, n_sev), m),
                severity=np.tile(sevs, m * F), arm=a, nll=nll.reshape(-1), correct=correct.reshape(-1))))
            if clean:
                frames.append(pd.DataFrame(dict(
                    dataset=dataset, backbone=backbone, run=run, fold=folds, image_id=np.arange(m),
                    family="clean", severity=0, arm=a, nll=0.3 + b + shift[a] + img_eff[a] + rng.normal(0, 0.05, m),
                    correct=correct_clean)))
    return validate_long(pd.concat(frames, ignore_index=True))

In [ ]:
#@title 8. Self-test (optional) { display-mode: "form" }
#@markdown Checks every method on simulated data with known answers: about 15 seconds in quick mode, 2 minutes in full.
RUN_SELFTEST = False  #@param {type:"boolean"}
SELFTEST_QUICK = True  #@param {type:"boolean"}


def selftest(quick=True):
    rng = np.random.default_rng(7)
    R = 0.25 if quick else 1.0
    results = []

    def check(name, cond, detail):
        results.append(bool(cond))
        print(f"  [{'pass' if cond else 'FAIL'}] {name}: {detail}")

    print("1. moment estimators recover known components (m = 900, F = 6; mean over replicates)")
    for s2a, s2b, s2e in [(0.0, 0.5, 1.0), (0.08, 0.0, 1.0), (0.3, 0.2, 0.5)]:
        est = []
        for _ in range(int(300 * R)):
            D = (rng.normal(0, math.sqrt(s2a), 6)[None] + rng.normal(0, math.sqrt(s2b), 900)[:, None]
                 + rng.normal(0, math.sqrt(s2e), (900, 6)))
            _, _, _, MSA, MSB, MSE = mean_squares(D)
            est.append(((MSA - MSE) / 900, (MSB - MSE) / 6, MSE))
        e = np.mean(est, 0)
        check(f"(s2a, s2b, s2e) = ({s2a}, {s2b}, {s2e})",
              abs(e[0] - s2a) < 0.03 + 0.25 * s2a and abs(e[1] - s2b) < 0.02 and abs(e[2] - s2e) < 0.02,
              f"recovered ({e[0]:.3f}, {e[1]:.3f}, {e[2]:.3f})")

    print("2. coverage of the exact SE-ratio CI, the generalized CI for rho_family and the full crossed CI")
    n = int(1000 * R)
    for s2a, s2b, s2e in [(0.05, 0.3, 1.0), (0.01, 0.0, 1.0), (0.2, 1.0, 0.5)]:
        true_R = math.sqrt((900 * s2a + s2e) / (6 * s2b + s2e)); true_rho = s2a / (s2a + s2b + s2e)
        cr = cg = cf = 0; ng = max(int(n * 0.5), 150)
        for k in range(n):
            D = 0.1 + (rng.normal(0, math.sqrt(s2a), 6)[None] + rng.normal(0, math.sqrt(s2b), 900)[:, None]
                       + rng.normal(0, math.sqrt(s2e), (900, 6)))
            tw = two_way(D, n_gpq=2000 if k < ng else 0, seed=k)
            cr += tw["se_ratio_lo"] <= true_R <= tw["se_ratio_hi"]
            cf += tw["ci_full"][0] <= 0.1 <= tw["ci_full"][1]
            if k < ng:
                cg += tw["rho_family_lo"] <= true_rho <= tw["rho_family_hi"]
        mc = 2 * math.sqrt(0.95 * 0.05 / n)
        check(f"SE ratio CI, true ratio {true_R:.2f}", abs(cr / n - 0.95) < mc + 0.005, f"coverage {cr / n:.3f} (n = {n})")
        check(f"rho_family GCI, true rho {true_rho:.3f}", cg / ng > 0.90, f"coverage {cg / ng:.3f} (n = {ng})")
        check("full crossed CI for the mean", cf / n > 0.92, f"coverage {cf / n:.3f} (n = {n})")

    print("3. tests of the family effect under a heteroskedastic null (no family effect)")
    sd = np.array([0.4, 0.6, 0.8, 1.0, 1.5, 2.2]); reps = int(400 * R)
    rej = np.zeros(3)
    for k in range(reps):
        common = rng.normal(0, 1, (900, 1))
        D = 0.1 + rng.normal(0, 0.3, (900, 1)) + (0.5 * common + math.sqrt(0.75) * rng.normal(0, 1, (900, 6))) * sd
        rej += [perm_test(D, n_perm=200, seed=k)["p"] < 0.05, hotelling_rm(D)["p_F"] < 0.05,
                wts_perm_test(D, n_perm=200, seed=k)["p"] < 0.05]
    r = rej / reps; mc = 2 * math.sqrt(0.05 * 0.95 / reps)
    print(f"       rejection at 0.05: plain permutation {r[0]:.3f}, Hotelling {r[1]:.3f}, "
          f"studentized permutation {r[2]:.3f} ({reps} replicates)")
    check("Hotelling T^2 holds its level", r[1] < 0.05 + mc, f"{r[1]:.3f}")
    check("studentized permutation holds its level", r[2] < 0.05 + mc, f"{r[2]:.3f}")
    a = np.zeros(6); a[4] = 0.25
    D = 0.1 + a[None] + rng.normal(0, 0.3, (900, 1)) + rng.normal(0, 1, (900, 6))
    pw = (perm_test(D, n_perm=2000)["p"], hotelling_rm(D)["p_F"], wts_perm_test(D, n_perm=2000)["p"])
    check("all three detect a single shifted family (shift 0.25 SD)", max(pw) < 0.01,
          "p = " + ", ".join(f"{x:.1e}" for x in pw))

    print("4. amplitude matching separates ladder calibration from mechanism")
    fams = [f"f{j}" for j in range(6)]; sevs = [1, 2, 3, 4, 5]
    loss = np.array([0.03, 0.05, 0.08, 0.11, 0.15, 0.20])[:, None] * np.arange(1, 6)[None]
    acc = 0.95 - loss
    for label, mech in [("ladder only", np.zeros(6)), ("ladder + mechanism", rng.normal(0, 0.15, 6))]:
        b = rng.normal(0, 0.3, (900, 1, 1))
        D3 = 0.8 * loss[None] + mech[None, :, None] * np.arange(1, 6)[None, None] / 3 + b \
            + rng.normal(0, 0.5, (900, 6, 5))
        un = two_way(D3.mean(2), n_gpq=0)
        am, _ = amplitude_matched(D3, acc, fams, sevs, targets=[0.15, 0.25], acc_clean=0.95, n_gpq=0)
        pmin = am["p_F_family"].min()
        print(f"       {label:20s} unmatched sigma_a {un['sigma_a']:.3f} (F test p {un['p_F_family']:.1e}); "
              f"matched sigma_a {', '.join(f'{x:.3f}' for x in am['sigma_a'])} (smallest p {pmin:.1e})")
        if label == "ladder only":
            check("ladder-only heterogeneity disappears after matching", un["p_F_family"] < 1e-6 and pmin > 0.001,
                  "unmatched significant, matched not")
        else:
            check("mechanism heterogeneity survives matching", pmin < 1e-3, "matched still significant")

    print("5. fold dependence")
    folds = np.repeat(np.arange(5), 180)
    for label, sf in [("no fold effect", 0.0), ("fold effect, true ICC 0.02", 0.0725)]:
        D = rng.normal(0, 1, (900, 6)) + rng.normal(0, sf, 5)[folds][:, None] + rng.normal(0, 0.3, (900, 1))
        fc = fold_clustered(D, folds, B=500)
        print(f"       {label:28s} ICC {fc['icc_fold']:.4f} [upper {fc['icc_fold_hi']:.4f}], design effect "
              f"{fc['design_effect']:.2f}; SE iid {fc['se_image_iid']:.4f}, cluster-robust "
              f"{fc['se_image_cluster_robust']:.4f}, two-stage {fc['se_image_two_stage']:.4f}")
    check("fold module reports a design effect", np.isfinite(fc["design_effect"]), "ok")

    print("6. end to end: a base run and one repeated partition")
    sim = simulate_long_table(runs=("G2", "G2-split1"), arms=("C2", "C4", "C5"), seed=3)
    cfg = SimpleNamespace(alpha=0.05, n_perm=500, n_perm_wts=300, n_gpq=2000, n_boot=300, seed=DEFAULT_SEED,
                          pairs=[], targets=None, amp_method="interp", split_tag="-split", light_splits=True,
                          out_dir=tempfile.mkdtemp(prefix="sdic_selftest_"), demo=True)
    res = run_all(sim, cfg)
    S = res["summary"]
    check("both comparisons in both runs, the partition in light mode",
          len(S) == 4 and int(S["light"].sum()) == 2, f"{len(S)} rows, {int(S['light'].sum())} light")
    sp = res["split"]
    check("repeated-partition table", sp is not None and len(sp) == 2 and (sp["n_splits"] == 2).all(),
          "2 comparisons x 2 partitions" if sp is not None else "missing")
    tb = res["tables"][(("SIM", "toy", "G2"), "C4 vs C2")]
    check("per-severity and amplitude tables", len(tb["per_severity"]) == 6 and tb["amplitude_matched"] is not None,
          f"{len(tb['per_severity'])} severity rows, {len(tb['amplitude_matched'])} amplitude targets")
    print("\nSELF-TEST", "PASSED" if all(results) else "FAILED")
    return all(results)


if RUN_SELFTEST:
    _t0 = time.time(); selftest(quick=SELFTEST_QUICK); print(f"({time.time() - _t0:.0f} s)")
else:
    print("Self-test skipped. Tick RUN_SELFTEST and run this cell to check the methods.")

In [ ]:
#@title 9. Load the per-image tables { display-mode: "form" }
if DATA_SOURCE == "demo":
    DF = simulate_long_table(runs=("G1", "G1-split1", "G1-split2"), arms=("C2", "C4", "C5", "C7q", "C8b"),
                             seed=1, dataset="DEMO", backbone="simulated")
    print("Demo: simulated data, not results.")
elif DATA_SOURCE == "upload":
    if not IN_COLAB:
        raise RuntimeError("upload works only in Colab; use drive or demo")
    _up = colab_files.upload()
    Path("/content/per_image").mkdir(exist_ok=True)
    _paths = []
    for _name, _data in _up.items():
        _p = Path("/content/per_image") / _name
        _p.write_bytes(_data); _paths.append(_p)
    DF = load_long(_paths)
elif DATA_SOURCE == "files":
    _paths = sorted(glob.glob(INPUT_GLOB))
    if not _paths:
        raise FileNotFoundError(f"no file matches {INPUT_GLOB}")
    print("\n".join(_paths))
    DF = load_long(_paths)
else:
    raise ValueError(f"unknown DATA_SOURCE {DATA_SOURCE!r}")
print(f"{len(DF):,} rows")
INV = inventory(DF)
display(INV)
if (INV["incomplete_arms"] != "none").any():
    print("Some arms do not cover every image x family x severity cell; comparisons with them will be skipped.")

In [ ]:
#@title 10. Run all analyses { display-mode: "form" }
_t0 = time.time()
RES = run_all(DF, CFG)
print(f"{len(RES['summary'])} comparisons in {time.time() - _t0:.0f} s; outputs in {RES['out']}")
_cols = ["dataset", "backbone", "run", "comparison", "mean", "se_ratio", "se_ratio_lo", "se_ratio_hi",
         "rho_family", "p_F_family", "perm_p", "T2_p_F", "wts_perm_p", "lofo_min_p_family", "design_effect"]
if len(RES["summary"]):
    display(RES["summary"].loc[~RES["summary"]["light"], _cols].round(4))

In [ ]:
#@title 11. Report { display-mode: "form" }
display(Markdown(RES["report"]))

In [ ]:
#@title 12. Inspect one comparison { display-mode: "form" }
#@markdown `RUN` as `dataset / backbone / run`, for example `NEU-CLS / resnet50.a1_in1k / G1`; empty takes the first run.
RUN = ""  #@param {type:"string"}
COMPARISON = "C4 vs C2"  #@param {type:"string"}
_keys = list(dict.fromkeys(k for k, _ in RES["tables"]))
_key = _keys[0] if not RUN.strip() else tuple(s.strip() for s in RUN.split("/"))
_tabs = RES["tables"].get((_key, COMPARISON.strip()))
if _tabs is None:
    print("Not found. Available:")
    for _k, _c in RES["tables"]:
        print("  ", " / ".join(_k), "|", _c)
else:
    _S = RES["summary"]
    _row = _S[(_S["dataset"] == _key[0]) & (_S["backbone"] == _key[1]) & (_S["run"] == _key[2])
              & (_S["comparison"] == COMPARISON.strip())]
    display(Markdown(f"### {' / '.join(_key)}: {COMPARISON.strip()}"))
    display(_row.T)
    for _name in ("lofo", "per_severity", "amplitude_matched", "family_by_severity_means"):
        if _tabs[_name] is not None:
            display(Markdown(f"**{_name.replace('_', ' ')}**"))
            display(_tabs[_name].round(4))

In [ ]:
#@title 13. Download the results as a zip { display-mode: "form" }
_base = Path("/content") if IN_COLAB else Path(tempfile.gettempdir())
_zip = shutil.make_archive(str(_base / "sdic_revision_results"), "zip", RES["out"])
print("archive:", _zip)
if IN_COLAB:
    colab_files.download(_zip)

## Where each result goes in the response

| Result | Placeholder in the response letter |
|---|---|
| Report, *Table 5 quantities, recomputed*, run G1 | R2.1, [PENDING: repository fix]: confirms every relabeled Table 5 row; put the same check in `sdic_tables_and_figures.ipynb` |
| Report, *tests of the family effect* | R2.3, [PENDING: permutation results for all comparisons] and [PENDING: results of both tests for all comparisons]; R3.2 |
| `lofo.csv` of each comparison | R2.6, [PENDING: leave-one-family-out results for all twelve comparisons, including standard-error ratios] |
| Summary rows against C5o | R2.6, [PENDING: oracle-class results] |
| Report, *fold dependence* and *repeated fold partitions* | R2.6 and R3.5, [PENDING: results of both checks] |
| `per_severity.csv` | R1.3, [PENDING: per-severity results]; R3.4 |
| `amplitude_matched.csv` | R3.4, [PENDING: per-severity and amplitude-matched results] |
| Report, generalized CI for $\rho_{family}$ and full crossed interval | Optional: extends Table 6 to every comparison (R2.2, R2.5) |

How to read each result is in the *Revision plan* tab of the response document.

## References

- Bates S, Hastie T, Tibshirani R (2024) Cross-validation: what does it estimate and how well does it do it? J Am Stat Assoc 119(546):1434–1445
- Burdick RK, Borror CM, Montgomery DC (2005) Design and Analysis of Gauge R&R Studies: Making Decisions with Confidence Intervals in Random and Mixed ANOVA Models. SIAM, Philadelphia
- Chung E, Romano JP (2013) Exact and asymptotically robust permutation tests. Ann Stat 41(2):484–507
- Friedrich S, Brunner E, Pauly M (2017) Permuting longitudinal data in spite of the dependencies. J Multivar Anal 153:255–265